# Generic Compute Engine

This notebook demos the generic compute engine end-to-end. It authenticates with ** Evo OAuth** and makes
**calls to the live compute discovery endpoint**
(`GET /compute/orgs/{org_id}/tasks`).

The thesis of the POC: a **fully generic** engine that reads the task catalogue **live from
discovery**, so a new platform task needs **no SDK release**. There is **no per-task
Python code** — every topic/task and every `run(...)` signature is synthesised from the
live schema. Execution delegates to `evo.compute.JobClient`. The engine is **async-native**, so calls are awaited (Jupyter supports top-level `await`).

What you'll see:
1. Authenticate  → a `manager` that *is* an `evo.common.IContext`.
2. `async with ComputeClient(manager) as client:` — discovery + auth happen on entry (**fail-fast**).
3. The dynamic, schema-driven namespace (`await client.<topic>.<task>.run(...)`).
4. Argument validation that happens **before** any network call.
5. The **raw** live catalogue via `DiscoveryClient` — the same in-package client the engine delegates discovery to (the capability still missing from the SDK).
6. **Live breadth vs. the generated stubs** — tasks the platform advertises *right now*
   that were never stubbed (including `feature_flag` / api-preview tasks).
7. (Optional) executing a task against Evo compute platform.

> Prereqs: `pip install -e packages/evo-sdk-common packages/evo-compute` and the notebook
> extras (`evo.notebooks`, `aiohttp`). Run this notebook from the `poc-engine/` folder
> so `poc_compute_engine` and `generate_stubs` import locally.

## 1. Authenticate

`ServiceManagerWidget` handles the OAuth login **and** organization/hub/workspace
selection, exactly as in the SDK's own code samples. The returned `manager` implements
`evo.common.IContext` (`get_connector()` + `get_org_id()`), which is all our engine needs.

Replace `client_id` / `redirect_url` with your Evo app credentials.

In [1]:
from evo.notebooks import ServiceManagerWidget

client_id = "native-pKoL7L5xCdUYeEzhjPuuvF2So"  # replace with your own client id from the Evo developer portal
redirect_url = "http://localhost:5050"  # replace with your own redirect url from the Evo developer portal

# OAuth issuer (IMS). Default is https://ims.bentley.com.
base_uri = "https://qa-ims.bentley.com"

# Evo Discovery base. The SDK appends /evo/identity/v2/discovery?service=evo, so this
# resolves to https://uat-discover.test.api.seequent.com/evo/identity/v2/discovery?service=evo
discovery_url = "https://uat-discover.test.api.seequent.com"

manager = await ServiceManagerWidget.with_auth_code(
    client_id=client_id,
    base_uri=base_uri,
    discovery_url=discovery_url,
    redirect_url=redirect_url,
    cache_location="./notebook-data",
).login()

/Users/amin.abedi/projects/EVO/evo-python-sdk/packages/evo-sdk-common/src/evo/notebooks/authorizer.py:108: UserWarning: The evo.notebooks.AuthorizationCodeAuthorizer is not secure, and should only ever be used in Jupyter notebooks in a private environment.
  warnings.warn(


ServiceManagerWidget(children=(VBox(children=(HBox(children=(Image(value=b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR…

In [ ]:
# Rich HTML display for Evo widgets (org/hub/workspace pickers above).
%load_ext evo.widgets

## 2. Construct the engine — fail-fast discovery on open

`ComputeClient` is **async-native**: opening it (here via `await ComputeClient.connect(manager)`, or `async with ComputeClient(manager) as client:`) makes the authenticated discovery call eagerly, so a missing/expired/unentitled token raises a typed SDK exception *here*, before any task namespace is handed out. There is no event-loop juggling and no `nest_asyncio` — every `run(...)` is awaited on the notebook's own loop.

In [2]:
from poc_compute_engine import ComputeClient

client = await ComputeClient.connect(manager)   # opens + runs the discovery call now (fail-fast)
client

<ComputeClient org='d844342d-a544-4a0b-b146-fc35036edf38' topics=['converter', 'cpt-to-borehole', 'geophysics', 'geostatistics', 'geotech', 'gtm', 'isosurfacing', 'test-faadefdcdbd', 'test-fecefabcedfadffd', 'vis-service']>

## 3. The live, schema-driven namespace

No per-task code: topics and tasks come straight from the live discovery response, and
each `.run` signature (parameter names, types, `Literal` enums, defaults) is synthesised
from the task's JSON schema.

In [3]:
import inspect

topics = [t for t in dir(client) if not t.startswith("_") and t not in ("connect", "aclose", "refresh")]
print("topics:", topics)

for topic in topics:
    ns = getattr(client, topic)
    print(f"  {topic}:", dir(ns))

# Show one synthesised signature.
topic = "geostatistics"
task = dir(getattr(client, topic))[0]
runner = getattr(getattr(client, topic), task).run
print(f"\nsignature of {topic}.{task}.run:\n  ", inspect.signature(runner))

topics: ['converter', 'cpt-to-borehole', 'geophysics', 'geostatistics', 'geotech', 'gtm', 'isosurfacing', 'test-faadefdcdbd', 'test-fecefabcedfadffd', 'vis-service']
  converter: ['detect_converter', 'ers_import', 'geosoft_grid_import', 'geosoft_surface_import', 'geosoft_voxel_import', 'image_import', 'obj_import', 'shp_import', 'ubc_import', 'vtk_import', 'xyz_import']
  cpt-to-borehole: ['cpt_reader', 'create_clusters', 'stratify']
  geophysics: ['gravity_terrain_correction', 'regular_2d_grid_to_trimesh']
  geostatistics: ['break_ties_gcp', 'conditional_tbsim_gcp', 'consim_gcp', 'continuous_dist_gcp', 'declustering', 'idw_gcp', 'knn_gcp', 'kriging_gcp', 'location_wise_gcp', 'loss_calculation_gcp', 'normal_score_gcp', 'profit_calculation_gcp', 'simulation_report_gcp']
  geotech: ['add_to_intersected_geometry', 'check_triangle_orientation', 'copy_geological_model_meshes', 'create_cross_section', 'create_geological_model_meshes', 'detect_degenerate_triangles', 'detect_duplicate_points',

## 4. Argument validation happens *before* the network call

Required parameters are enforced from the schema, so a bad call fails locally instead of
burning a round-trip.

In [4]:
try:
    await runner()  # await drives the coroutine body, where validation runs (no network reached)
except TypeError as exc:
    print("caught locally (no network):", exc)

caught locally (no network): break-ties-gcp.run() missing required parameter(s): ['neighborhood', 'source', 'target']


## 5. A specialized typed runner (the override seam)

Most tasks ride the generic engine, but a task can opt into a **hand-written, fully-typed runner** by dropping a module at `poc_compute_engine/overrides/<topic>/<task>.py`. The engine auto-discovers it by convention and routes to it transparently — same `client...run(...)` DX.

`geostatistics/kriging-gcp` has such an override ([`overrides/geostatistics/kriging_gcp.py`](./poc_compute_engine/overrides/geostatistics/kriging_gcp.py)), so `client.geostatistics.kriging_gcp` resolves to a `KrigingGcpRunner` with bespoke validation, a `mean` parameter absent from the discovery schema, and result helpers (`summary()`, `portal_url()`). The other geostatistics tasks stay generic.

In [5]:
import inspect

kriging = client.geostatistics.kriging_gcp
declustering = client.geostatistics.declustering
print("kriging      ->", type(kriging).__name__)        # KrigingGcpRunner (specialized override)
print("declustering ->", type(declustering).__name__)   # _TaskProxy (generic)
print("kriging.run  ->", inspect.signature(kriging.run))  # note the override-only `mean` param

# Override-only validation the JSON Schema can't express — runs BEFORE any network call
# (awaiting drives the coroutine body up to the validation, no HTTP reached):
try:
    await kriging.run(source="grade", target="kriged", variogram="vario-1", kriging_type="simple")
except ValueError as exc:
    print("validation:", exc)

kriging      -> KrigingGcpRunner
declustering -> _TaskProxy
kriging.run  -> (*, source: 'str', target: 'str', variogram: 'str', kriging_type: "Literal['simple', 'ordinary']" = 'ordinary', max_samples: 'int' = 20, mean: 'float | None' = None, preview: 'bool' = True) -> 'KrigingGcpResult'
validation: simple kriging requires `mean` to be specified


In [6]:
# The override also owns the OUTPUT side: a hand-curated, fully-typed result with
# helpers the generic TaskResult can't synthesise. Real execution needs live object
# references (see the optional execute cell below), so here we hydrate the typed result
# from a representative discovery-shaped payload to show the surface it exposes.
from poc_compute_engine.overrides.geostatistics.kriging_gcp import KrigingGcpResult

sample_payload = {
    "message": "Kriging completed.",
    "target": {
        "reference": "https://hub.evo/objects/2f1c-0000-abcd",
        "name": "grade_estimate_grid",
        "description": None,
        "schema_id": "regular-3d-grid/1.2.0",
        "attribute": {
            "reference": "cell_attributes[?name=='kriged_grade']",
            "name": "kriged_grade",
        },
    },
}
result = KrigingGcpResult(sample_payload, kriging_type="ordinary")

print("message      :", result.message)
print("target name  :", result.target.name)
print("attribute    :", result.target.attribute.name)
print("summary()    :", result.summary())       # override-only helper
print("portal_url() :", result.portal_url())    # override-only helper
print("to_dataframe :", result.target.to_dataframe())

message      : Kriging completed.
target name  : grade_estimate_grid
attribute    : kriged_grade
summary()    : ordinary kriging wrote attribute 'kriged_grade' to 'grade_estimate_grid'
portal_url() : https://portal.mock.evo/objects/2f1c-0000-abcd
to_dataframe : Table(rows=2, columns=['i', 'j', 'k', 'value'])


## 6. The raw live catalogue via `DiscoveryClient`

`evo.compute` has a `JobClient` for *executing* tasks but **no discovery client** — its `TasksApi` only exposes `execute_task`. Listing tasks is the missing capability. The engine delegates discovery to `poc_compute_engine.DiscoveryClient` (the symmetric twin of `JobClient`); here we use that **same** client directly to show the raw catalogue — including api-preview `feature_flag` tasks — over the authenticated context.

In [7]:
from poc_compute_engine import DiscoveryClient

tasks = await DiscoveryClient.from_context(manager).list_tasks()
print(f"{len(tasks)} tasks advertised by the platform:\n")
for t in sorted(tasks, key=lambda s: (s["topic"], s["name"])):
    flag = f"  [api-preview: {t['feature_flag']}]" if t.get("feature_flag") else ""
    print(f"  {t['topic']}/{t['name']:<22} v{t.get('version','?')}{flag}")

94 tasks advertised by the platform:

  converter/detect-converter       v0.1.13  [api-preview: converter-preview]
  converter/ers-import             v0.1.13  [api-preview: converter-preview]
  converter/geosoft-grid-import    v0.1.13  [api-preview: converter-preview]
  converter/geosoft-surface-import v0.1.13  [api-preview: converter-preview]
  converter/geosoft-voxel-import   v0.1.13  [api-preview: converter-preview]
  converter/image-import           v0.1.13  [api-preview: converter-preview]
  converter/obj-import             v0.1.13  [api-preview: converter-preview]
  converter/shp-import             v0.1.13  [api-preview: converter-preview]
  converter/ubc-import             v0.1.13  [api-preview: converter-preview]
  converter/vtk-import             v0.1.13  [api-preview: converter-preview]
  converter/xyz-import             v0.1.13  [api-preview: converter-preview]
  cpt-to-borehole/cpt-reader             v0.1.28  [api-preview: cpt-to-borehole-preview]
  cpt-to-borehole/create-c

## 7. Live breadth vs. the generated stubs

The `.pyi` stubs are generated **offline** from a point-in-time snapshot in `poc_compute_engine/schemas/`
(see `generate_stubs.py`). The runtime is always live, so the platform typically advertises
**more** tasks than the snapshot knows — those run fine through the engine today, they just
aren't statically typed until stubs are regenerated. This is the generic-vs-codegen trade:
**total runtime breadth, point-in-time static breadth**.

In [8]:
from generate_stubs import _load_bundled_specs

stubbed = {(s["topic"], s["name"]) for s in _load_bundled_specs()}
live = await DiscoveryClient.from_context(manager).task_keys()

print("stubbed (typed in __init__.pyi):")
for k in sorted(stubbed):
    print("   ", "/".join(k))

print("\nadvertised live but NOT stubbed (runnable now, no SDK release needed):")
for k in sorted(live - stubbed):
    print("   ", "/".join(k))

missing_stub = stubbed - live
if missing_stub:
    print("\nstubbed but not currently advertised:", sorted(missing_stub))

stubbed (typed in __init__.pyi):
    geostatistics/declustering
    geostatistics/kriging-gcp
    geostatistics/normal-score-gcp

advertised live but NOT stubbed (runnable now, no SDK release needed):
    converter/detect-converter
    converter/ers-import
    converter/geosoft-grid-import
    converter/geosoft-surface-import
    converter/geosoft-voxel-import
    converter/image-import
    converter/obj-import
    converter/shp-import
    converter/ubc-import
    converter/vtk-import
    converter/xyz-import
    cpt-to-borehole/cpt-reader
    cpt-to-borehole/create-clusters
    cpt-to-borehole/stratify
    geophysics/gravity-terrain-correction
    geophysics/regular-2d-grid-to-trimesh
    geostatistics/break-ties-gcp
    geostatistics/conditional-tbsim-gcp
    geostatistics/consim-gcp
    geostatistics/continuous-dist-gcp
    geostatistics/idw-gcp
    geostatistics/knn-gcp
    geostatistics/location-wise-gcp
    geostatistics/loss-calculation-gcp
    geostatistics/profit-calculation-g

## 7.5 Reference resolution — friendly values → wire payload

The generic engine no longer *stubs* reference resolution. A single, task-agnostic
`ReferenceResolver` reads each task's schema annotations (the closed vocabulary:
`reference_to`, `supported_schemas`, `attribute_from`, `attribute_path`, `target`,
`discriminator`) and turns friendly Python values into the exact wire payload the
platform expects — objects → validated URLs, attributes → JMESPath, targets →
create/update shapes, unions → the right branch.

Resolving an *attribute* needs the **owning object's** `schema_id` (a `pointset` keeps
attributes under `locations.attributes`, a `block-model` under `attributes`), so the
resolver takes an injectable **`ObjectLoader`**: the real authenticated load in
production, or — as below — an `ObjectHandle` that carries its `schema_id` inline so the
demo resolves a real payload **without a network round-trip**.


In [11]:
from poc_compute_engine.resolver import (
    ReferenceResolver, ObjectHandle, AttributeRef, CreateAttr, LoadedObject,
)

# A loader is required, but ObjectHandles below carry their own schema_id, so it is
# never actually called here (real notebooks pass an authenticated, SDK-backed loader).
class _UnusedLoader:
    async def load(self, handle): raise AssertionError("handles carry schema_id inline")

resolver = ReferenceResolver(_UnusedLoader())

spec = client._spec("geostatistics", "kriging-gcp")  # the live schema discovery returned
pointset = ObjectHandle(reference="https://hub/objects/ps-1", schema_id="pointset/1.2.0")
grid     = ObjectHandle(reference="https://hub/objects/bm-1", schema_id="block-model/1.0.0")
vario    = ObjectHandle(reference="https://hub/objects/vg-1", schema_id="variogram/1.1.0")

payload = await resolver.resolve(spec, {
    "source": {"object": pointset, "attribute": AttributeRef("grade")},
    "target": {"object": grid, "attribute": CreateAttr("kriged_grade")},
    "variogram": vario,
    "kriging_method": {"type": "ordinary"},          # discriminated union -> branch by `type`
})

import json
print(json.dumps(payload, indent=2))
# Note how each friendly value became wire-correct:
#   source.object    -> the pointset URL
#   source.attribute -> "locations.attributes[?name=='grade']"  (pointset container)
#   target.attribute -> {"operation": "create", "name": "kriged_grade"}
#   variogram        -> validated against supported_schemas, then its URL


{
  "source": {
    "object": "https://hub/objects/ps-1",
    "attribute": "locations.attributes[?name=='grade']"
  },
  "target": {
    "object": "https://hub/objects/bm-1",
    "attribute": {
      "operation": "create",
      "name": "kriged_grade"
    }
  },
  "variogram": "https://hub/objects/vg-1",
  "kriging_method": {
    "type": "ordinary"
  }
}


## 7.6 Online end-to-end — how *your* inputs map to *the schema*

§7.5 resolved a payload offline. This section does it **online, against a real
task** to show the full generic path: you write friendly Python, and the engine —
using the schema that **live discovery** returned into `client` — decides the wire
shape for every field. Nothing here is task-specific code; swap `declustering`
for any discovered task and the same machinery applies.

We use **`declustering`** because it is a *generic* task (no hand-written override),
so what you see is purely schema-driven. Its parameters (from discovery):
`source`, `grid`, `target`, `neighborhood`, and optional `power`.

In [12]:
# The LIVE schema discovery returned for this task (already fetched when `client` opened).
spec = client._spec("geostatistics", "declustering")
assert spec is not None, "declustering not in the live catalogue for this environment"

# ---- The friendly inputs a user writes (no URLs, no JMESPath, no operation dicts) ----
from poc_compute_engine.resolver import ObjectHandle, AttributeRef, CreateAttr

# In a real notebook these come from evo.objects (loaded/typed objects or their attributes);
# here we use ObjectHandles that carry schema_id inline so the *mapping* is visible with no
# network call. Section further below shows the fully authenticated variant.
friendly = {
    "source": {"object": ObjectHandle("https://hub/objects/ps-1", "pointset/1.2.0"),
               # (declustering's `source` has no attribute slot — weights come from the grid)
              },
    "grid":   {"object": ObjectHandle("https://hub/objects/grid-1", "regular-masked-3d-grid/1.2.0")},
    "target": {"object": ObjectHandle("https://hub/objects/grid-1", "regular-masked-3d-grid/1.2.0"),
               "attribute": CreateAttr("decluster_weight")},
    "neighborhood": {"ellipsoid": {"ranges": [200, 150, 100]}, "max_samples": 20},
    "power": None,                       # explicit null -> KNN mode (missing != null)
}

# ---- A tiny schema walker: for each field, report which annotation drives resolution ----
ANNOTATIONS = ("reference_to", "supported_schemas", "attribute_from",
               "attribute_path", "target", "discriminator")

def describe(node, path=""):
    if not isinstance(node, dict):
        return
    hits = {a: node[a] for a in ANNOTATIONS if a in node}
    if hits:
        for a, v in hits.items():
            s = json.dumps(v)
            print(f"  {path or '<root>':<28} {a:<17} -> {s[:70]}{'…' if len(s)>70 else ''}")
    for k, v in node.get("properties", {}).items():
        describe(v, f"{path}/{k}")
    for i, b in enumerate(node.get("oneOf", [])):
        describe(b, f"{path}[oneOf{i}]")

print("HOW THE RESOLVER READS declustering's SCHEMA (closed vocabulary only):\n")
describe(spec["parameters"], "")
print("\nrequired:", spec["parameters"].get("required"))

HOW THE RESOLVER READS declustering's SCHEMA (closed vocabulary only):

  /grid/object                 reference_to      -> "geoscience-object"
  /grid/object                 supported_schemas -> ["pointset/[>=1.2,<2]", "block-model/[>=1.0,<2]", "regular-3d-grid/[>=…
  /source/object               reference_to      -> "geoscience-object"
  /source/object               supported_schemas -> ["pointset/[>=1.2,<2]", "block-model/[>=1.0,<2]", "regular-3d-grid/[>=…
  /target/attribute            target            -> "attribute"
  /target/attribute[oneOf1]/reference reference_to      -> "attribute"
  /target/attribute[oneOf1]/reference attribute_from    -> "2/object"
  /target/attribute[oneOf1]/reference attribute_path    -> {"block-model/[>=1.0,<2]": ["attributes[?attribute_type=='Float64']"],…
  /target/object               reference_to      -> "geoscience-object"
  /target/object               supported_schemas -> ["pointset/[>=1.2,<2]", "block-model/[>=1.0,<2]", "regular-3d-grid/[>=…

req

In [13]:
# ---- Now actually resolve the friendly inputs into the wire payload ----
# `client._resolver` is the SAME instance `run(...)` uses internally; calling it here just
# lets us SEE the payload the engine would submit. (ObjectHandles carry schema_id, so no
# network is needed; with real objects the injected ObjectLoader fetches schema_id online.)
payload = await client._resolver.resolve(spec, friendly)

print("WIRE PAYLOAD the engine will POST to the compute platform:\n")
print(json.dumps(payload, indent=2))

print("""
Read the mapping top-to-bottom:
  source.object   friendly ObjectHandle      -> validated URL      (reference_to + supported_schemas)
  grid.object     friendly ObjectHandle      -> validated URL
  target.object   friendly ObjectHandle      -> validated URL
  target.attribute CreateAttr("...")          -> {"operation":"create","name":"..."}  (target oneOf)
  neighborhood    passed through structurally (no reference annotations -> literal)
  power           None kept as null           (schema declares power nullable -> KNN mode)
""")

WIRE PAYLOAD the engine will POST to the compute platform:

{
  "source": {
    "object": "https://hub/objects/ps-1"
  },
  "grid": {
    "object": "https://hub/objects/grid-1"
  },
  "target": {
    "object": "https://hub/objects/grid-1",
    "attribute": {
      "operation": "create",
      "name": "decluster_weight"
    }
  },
  "neighborhood": {
    "ellipsoid": {
      "ranges": [
        200,
        150,
        100
      ]
    },
    "max_samples": 20
  },
  "power": null
}

Read the mapping top-to-bottom:
  source.object   friendly ObjectHandle      -> validated URL      (reference_to + supported_schemas)
  grid.object     friendly ObjectHandle      -> validated URL
  target.object   friendly ObjectHandle      -> validated URL
  target.attribute CreateAttr("...")          -> {"operation":"create","name":"..."}  (target oneOf)
  neighborhood    passed through structurally (no reference annotations -> literal)
  power           None kept as null           (schema declares power 

In [ ]:
# ---- The real thing: one call. Resolution + submit + poll all happen inside run(...) ----
# Uncomment and supply REAL, authenticated inputs (typed objects/attributes from evo.objects,
# or ObjectHandles pointing at real object URLs in THIS workspace). The engine will:
#   1) validate args against the live schema (fail-fast, before any network),
#   2) call client._resolver.resolve(...) — objects->URLs, attributes->JMESPath, target->create/update,
#      fetching each object's schema_id online via the authenticated ObjectLoader,
#   3) submit through the real evo.compute.JobClient (submit -> poll -> results),
#   4) hydrate a TaskResult from the schema's `results` block.
#
result = await client.geostatistics.declustering.run(
    source={"object": "<real pointset object id/url>"},
    grid={"object": "<real grid object id/url>"},
    target={"object": "<real grid object id/url>", "attribute": CreateAttr("decluster_weight")},
    neighborhood={"ellipsoid": {"ranges": [200, 150, 100]}, "max_samples": 20},
    preview=True,
)
print(result.message)
result   # pretty-printed, schema-hydrated result

## 8. (Optional) Execute a task 

Execution is wired to the existing `evo.compute.JobClient` (submit → poll → results),
with references resolved for real by the `ReferenceResolver` (§7.5). Running a real task
needs **real** objects (a source attribute, a target object, a variogram, ...). Building
those inputs is out of scope for this engine demo — see the SDK's full
`code-samples/.../running-kriging-compute.ipynb` for an end-to-end example that creates a
PointSet, Variogram and BlockModel first.

> With a real (authenticated) `ObjectLoader`, pass typed/loaded objects or `ObjectHandle`s
> directly as `source`/`target`/`variogram` and the engine resolves them before submitting.


In [ ]:
# Uncomment and supply REAL references to submit a job through evo.compute.JobClient:
#
# result = await client.geostatistics.kriging_gcp.run(
#     source="<real source attribute reference>",
#     target="<real target attribute reference>",
#     variogram="<real variogram object id>",
# )
# print(result.message)
# result.target.to_dataframe()

## 8.5 Running Kriging **live** — the generic engine, zero bespoke code

Kriging ships with a specialized override (§5) for its *typed DX*, but the estimate itself
needs **no bespoke runtime**: the generic path resolves its inputs and submits through the
real `evo.compute.JobClient` exactly like any other task. Below is a genuinely live run —
friendly Python in, a schema-hydrated result out — routed through the pure generic engine.

The inputs must be **real geoscience objects in your workspace**: a source *attribute*, a
target *grid*, and a *variogram*. Building those is `evo.objects` territory — load existing
ones by UUID (shown here) or create them from a DataFrame as in the SDK's
`code-samples/.../running-kriging-compute.ipynb`. Because this **writes an attribute and
consumes compute**, it is gated behind `RUN_LIVE_KRIGING` — set it to `True` to actually
run.

In [ ]:
# Set True to load real objects, resolve their references, and submit a real kriging job.
RUN_LIVE_KRIGING = False

if RUN_LIVE_KRIGING:
    from evo.objects.typed import object_from_uuid

    # --- Real objects in THIS workspace (replace with your own UUIDs) ---------------
    # (Alternatively create them from DataFrames via PointSet.create / RegularMasked3DGrid.create
    #  / Variogram.create — see the SDK's running-kriging-compute.ipynb.)
    source_pointset = await object_from_uuid(manager, "<source pointset uuid>")
    target_grid     = await object_from_uuid(manager, "<target grid uuid>")
    variogram       = await object_from_uuid(manager, "<variogram uuid>")

    print("source   :", source_pointset.name)
    print("target   :", target_grid.name)
    print("variogram:", variogram.name)
else:
    print("RUN_LIVE_KRIGING is False — set it True (with real UUIDs) to run kriging live.")

In [ ]:
if RUN_LIVE_KRIGING:
    from poc_compute_engine.engine import _make_run
    from poc_compute_engine.resolver import CreateAttr

    # Build the GENERIC run for kriging — the *same* code path `_make_run` gives every
    # task (resolve references -> submit via JobClient -> hydrate the result). No override.
    spec = client._spec("geostatistics", "kriging-gcp")
    assert spec is not None, "kriging-gcp not in the live catalogue for this environment"
    kriging_run = _make_run(client, spec)

    # Friendly inputs — the resolver turns each into the wire payload (see §7.6):
    #   source  : a typed Attribute  -> {object: <pointset url>, attribute: JMESPath}
    #   target  : a grid + CreateAttr -> {object: <grid url>, attribute: {operation: create, ...}}
    #   variogram: a typed object     -> validated against supported_schemas, then its URL
    result = await kriging_run(
        source=source_pointset.attributes["grade"],   # typed-attribute shorthand
        target={"object": target_grid, "attribute": CreateAttr("kriged_grade")},
        variogram=variogram,
        kriging_method={"type": "ordinary"},          # discriminated union -> branch by `type`
        neighborhood={
            "ellipsoid": {
                "ellipsoid_ranges": {"major": 200.0, "semi_major": 150.0, "minor": 100.0},
                "rotation": {"dip_azimuth": 100.0, "dip": 65.0, "pitch": 75.0},
            },
            "max_samples": 20,
        },
        preview=True,
    )

    print(result)                          # TaskResult, hydrated from the schema `results`
    df = result.target.to_dataframe()      # the kriged grid, as a DataFrame
    display(df.head())

## 9. Cleanup

Close the engine (closes the transport; the notebook's own event loop is left running).

In [ ]:
await client.aclose()